# 階段二:預測性維護基準(可重現起點)

用本平台的**合成**資料訓練故障分類 + RUL 迴歸,證明資料學得起來,並當你改進的起點。

> ⚠ 合成資料:模型學的是「假設的退化物理」,適合教 ML 工作流程,不保證遷移真實設備(domain gap)。

先備(在專案根目錄、用 venv 的 python / kernel):
```
python tools/generate_dataset.py --sim-days 120 --step-min 10 --out dataset
python -m pip install -r student_kit/requirements-ml.txt
```


In [ ]:
import sys, os
sys.path.append('student_kit')          # 借用 p4_train_baseline 的資料/特徵函式
import numpy as np, pandas as pd
from p4_train_baseline import load_frames, feature_columns, add_rolling, rmse
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import f1_score, roc_auc_score, mean_absolute_error, r2_score


## 1) 載入 + 特徵工程(防洩漏)
依機台切 train/test;滾動特徵在每個 run-to-failure 循環內算。換 `pattern` 可試別的設備型別。


In [ ]:
pattern = 'cnc-*.csv'      # 試試 'wt-*.csv' / 'comp-*.csv'
files, frames = load_frames(pattern, 'dataset')
devices = [os.path.splitext(os.path.basename(f))[0] for f in files]
n_test = max(1, round(len(devices)*0.3))
train_dev, test_dev = devices[:-n_test], devices[-n_test:]
raw = pd.concat(frames, ignore_index=True)
data, feats = add_rolling(raw, feature_columns(raw), window=12)
data = data.dropna(subset=['ttf_sim_s'])
tr = data[data.device.isin(train_dev)]; te = data[data.device.isin(test_dev)]
print('train rows', len(tr), '| test rows', len(te), '| features', len(feats))
print('test 機台(沒看過):', test_dev)


## 2) 故障分類:24h 內會不會故障


In [ ]:
clf = RandomForestClassifier(n_estimators=120, max_depth=14, min_samples_leaf=20,
                             class_weight='balanced', n_jobs=-1, random_state=0)
clf.fit(tr[feats].fillna(0), tr.fail_within_24h.astype(int))
proba = clf.predict_proba(te[feats].fillna(0))[:,1]
print('F1     ', round(f1_score(te.fail_within_24h, proba>=0.5), 3))
print('ROC-AUC', round(roc_auc_score(te.fail_within_24h, proba), 3))


## 3) RUL 迴歸:剩餘壽命(小時)


In [ ]:
ytr = tr.ttf_sim_s.to_numpy()/3600; yte = te.ttf_sim_s.to_numpy()/3600
reg = RandomForestRegressor(n_estimators=120, max_depth=16, min_samples_leaf=20,
                            n_jobs=-1, random_state=0)
reg.fit(tr[feats].fillna(0), ytr)
pred = reg.predict(te[feats].fillna(0))
print('MAE (h)', round(mean_absolute_error(yte, pred), 2), '| R2', round(r2_score(yte, pred), 3))


## 4) 看一個測試循環:預測 RUL vs 真實 + 振動


In [ ]:
import matplotlib.pyplot as plt
dev, cyc = test_dev[0], te[te.device==test_dev[0]].cycle_id.iloc[len(te[te.device==test_dev[0]])//2]
g = te[(te.device==dev) & (te.cycle_id==cyc)].sort_values('sim_t')
h = g.sim_t/3600
plt.figure(figsize=(8,4))
plt.plot(h, g.ttf_sim_s/3600, label='真實 RUL', color='#2f7a4f')
plt.plot(h, reg.predict(g[feats].fillna(0)), '--', label='預測 RUL', color='#b5743a')
plt.xlabel('sim 時間 (h)'); plt.ylabel('RUL (h)'); plt.legend(); plt.title(f'{dev} cycle {cyc}'); plt.show()


## 5) 換你改進
- 更好的特徵:頻域、循環內趨勢斜率、多訊號交互。
- 更強的模型:GradientBoosting / XGBoost / LSTM / survival models。
- **閉環**:把模型接成服務,打 `POST /api/predictions` 在故障前告警,比 lead time(見 `student_kit/p3_predictor.py`)。
- 處理感測器故障(`is_sensor_fault`):分辨「設備壞 vs 感測器壞」是進階題。
